In [1]:
from glob import glob
import pandas as pd
import os
import subprocess
import shlex
import shutil
from collections import defaultdict
from itertools import product
from tqdm.auto import tqdm

os.makedirs('chipseq_new', exist_ok=True)
os.makedirs('ghtselex_new', exist_ok=True)
os.makedirs('chipseq', exist_ok=True)
os.makedirs('ghtselex', exist_ok=True)
os.makedirs('chipseq+ghtselex', exist_ok=True)

In [2]:
approved = set(pd.read_excel('chipseq_metadata.xlsx', sheet_name=3, header=0).query('`Success type` == "Approved"')['File_name'])
chipseq_tfs = set()
for file in approved:
    tf = file.split('_')[0]
    chipseq_tfs.add(tf)
    new_name = 'chipseq/' + tf + '.bed'
    old_name = 'chipseq_raw/' + file
    shutil.copy(old_name, new_name)
    

In [3]:
ghtselex_tfs = defaultdict(list)
for file in tqdm(glob('ghtselex_raw/*')):
    tf = file.split('/')[-1].split('.')[0].split('_')[0]
    new_path = file.replace('raw', 'new')
    cmd = f' tail -n +2 {file} > {new_path}'
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ghtselex_tfs[tf].append(new_path)

for tf, files in tqdm(ghtselex_tfs.items()):
    files_str = ' '.join(files)
    cmd = f'cat {files_str} | cut -f -3 | bedtools sort | bedtools merge > ghtselex/{tf}.bed'
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)

  0%|          | 0/206 [00:00<?, ?it/s]

  0%|          | 0/179 [00:00<?, ?it/s]

In [6]:
tfs = chipseq_tfs | set(ghtselex_tfs.keys())

for tf in tqdm(tfs):
    files = glob(f'chipseq/{tf}.bed') + glob(f'ghtselex/{tf}.bed')
    files = ' '.join((f'{file}' for file in files))
    cmd = f'cat {files} | cut -f -3 | bedtools sort | bedtools merge > chipseq+ghtselex/{tf}.bed'
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)

  0%|          | 0/216 [00:00<?, ?it/s]